# 02 - Live view: see the ACC work, and watch the experiments run
**Part A** is an interactive illustration (change the scenario, number of agents and fixed window, and the plot updates).
**Parts B-D** run the real RQ2 experiments in the background and redraw a live progress view every few seconds.

Everything is resumable: if Colab disconnects or you stop a cell, run it again and it continues where it stopped.
No GPU needed (Runtime -> Change runtime type -> CPU is fine).

In [ ]:
# 1) Setup: Drive, repo, dependencies   (run once per session)
import os, sys, subprocess, base64, tempfile
from pathlib import Path

REPO_URL = "https://github.com/notanut/eem-acc-skripsi.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/eem-acc-skripsi"
    DRIVE_ROOT = "/content/drive/MyDrive/Skripsi"
    def git(*args, cwd=None):
        subprocess.run(["git", *args], cwd=cwd, check=True)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        head = lambda: subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout
        before = head()
        git("pull", "--ff-only", cwd=REPO_DIR)
        if head() != before and "eem_acc" in sys.modules:
            print("NEW CODE WAS PULLED, but the old version is still loaded.\n"
                  ">>> Runtime -> Restart session, then run this cell again. <<<")
    else:
        git("clone", REPO_URL, REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt", "-e", REPO_DIR], check=True)
else:                                   # running outside Colab (local test): use this checkout + a temp folder
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
    DRIVE_ROOT = tempfile.mkdtemp(prefix="eem_acc_")
# Make the code importable NOW: an editable pip install only takes effect after a runtime restart in Colab.
import importlib
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
importlib.invalidate_caches()
if f"{REPO_DIR}/experiments" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/experiments")
os.environ["EEM_ACC_DRIVE_ROOT"] = DRIVE_ROOT
from eem_acc.utils import Paths, set_seed
set_seed(42); Paths().ensure()
print("repo:", REPO_DIR, "\nresults folder:", DRIVE_ROOT)

## Part A - Interactive illustration (not a result)
ACC (using its *start values*, not tuned ones) against a fixed window, on one seed. Pick a scenario, the number of agents (M) and the fixed window.
Top: the window each method allows. Middle: how many attempts per second were overload signals (429/503/client timeouts). Bottom: p95 server latency against the target (dashed).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from eem_acc import scenarios as S
from eem_acc.viz import demo_figure

def show(scenario="CAP-drop", m_agents=32, fixed_w=16):
    fig = demo_figure(scenario, m_agents, seed=0, fixed_w=fixed_w)
    plt.show()

try:
    from ipywidgets import interact, Dropdown
    interact(show,
             scenario=Dropdown(options=list(S.SCENARIOS), value="CAP-drop", description="scenario"),
             m_agents=Dropdown(options=list(S.M_LEVELS), value=32, description="agents M"),
             fixed_w=Dropdown(options=[4, 8, 16, 24, 32, 64], value=16, description="fixed W"));
except ImportError:
    show()

## Part B - Watch the dev phase run live
16,200 simulated runs (design v2: 243 ACC settings + 12 fixed windows + 15 token-bucket rates, 4 scenarios, 3 agent counts, 5 seeds; each run = 120 s of simulated time). About 2 hours on a fast machine; plan for roughly 4-6 hours on Colab's CPU (resumable: if Colab disconnects, run the cell again). The ETA is rough: the first runs are the light 8-agent ones, so it grows as heavier runs start.
Top: progress and ETA. Tiles: runs finished per scenario and agent count. Table: **partial** ranking of methods from the runs finished so far (it only becomes meaningful when the phase is complete).

Set `LIMIT` to a small number (e.g. 60) for a quick trial; keep `None` for the full run. If you stop the cell, run it again to continue.

Results go to `Skripsi/rq2_runs/<RUN_NAME>/` on your Drive (set `RUN_NAME` in the cell below).

In [ ]:
import json, threading, time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import run_rq2
from eem_acc.analysis import load_results, score_methods
from eem_acc.monitor import snapshot
from eem_acc.viz import monitor_figure

# Each experiment version gets its OWN results folder, so trial runs never mix with (or block) real runs.
# Use a new RUN_NAME for a trial or after the simulator/design changes; reuse the same name to resume.
RUN_NAME = "trial"
ROOT = Path(DRIVE_ROOT) / "rq2_runs" / RUN_NAME
LIMIT = None          # e.g. 60 for a quick trial
print("results for this run go to:", ROOT)
# background workers survive re-running this cell (stopping the cell only stops the DISPLAY, not the run)
_WORKERS = globals().get("_WORKERS", {})

def watch(phase, frozen=None, limit=None, refresh=5.0):
    key = (phase, str(ROOT))
    others = [k for k, v in _WORKERS.items() if k != key and v["thread"].is_alive()]
    if others:
        print("WARNING: other runs are still working in the background:", others,
              "\nThey keep using CPU. Restart the runtime (Runtime -> Restart session) to stop them.")
    w = _WORKERS.get(key)
    if w is None or not w["thread"].is_alive():
        w = {"err": []}
        def work():
            try:
                run_rq2.run_phase(phase, ROOT, limit, frozen)
            except BaseException as e:
                w["err"].append(e)
        w["thread"] = threading.Thread(target=work, daemon=True)
        w["thread"].start()
        _WORKERS[key] = w
    udir = ROOT / "results" / "rq2" / phase / "units"
    while True:
        alive = w["thread"].is_alive()
        snap = snapshot(ROOT, phase, frozen)
        clear_output(wait=True)
        fig = monitor_figure(snap); plt.show()
        df = load_results(udir) if udir.exists() and any(udir.glob("*.json")) else None
        if df is not None and len(df):
            cols = ["tag", "score", "mean_slo_violation", "mean_call_success", "n_cells"]
            print(f"Partial ranking from {len(df)} finished runs (final only when the phase is complete):")
            display(score_methods(df)[cols].head(8).round(3))
        if not alive:
            break
        time.sleep(refresh)
    if w["err"]:
        raise w["err"][0]
    print("finished.")

watch("dev", limit=LIMIT)

## Part C - Select the configuration (dev results only)
Picks the ACC settings, fixed window and token-bucket rate from the **dev** runs using the approved rule (design v2: maximise normalised goodput over the loaded cells, subject to SLO violation <= 5%, call success >= 99% and throttled attempts <= 10% in EVERY loaded cell). Also stores the per-scenario best fixed windows, the Pareto front and the leave-one-scenario-out check.
It refuses to run until the dev phase is complete, and it never relaxes the constraints on its own.

In [ ]:
FORCE = False   # set True ONLY if select reported code 6 and you really want to replace the old frozen file
code_ = run_rq2.do_select(ROOT, force=FORCE)
print("exit code:", code_)
print("0 = chosen and saved | 2 = dev phase incomplete (expected after a LIMIT trial) | 3 = no feasible candidate\n"
      "4 = test results already exist | 5 = dev results from another code version/grid\n"
      "6 = configs/rq2_frozen.json in this Colab clone holds a DIFFERENT selection (e.g. from another RUN_NAME):\n"
      "    set FORCE = True and re-run this cell to replace it with this run's selection")


## Part D - Watch the test phase run live
About 1,560 runs with the frozen configuration and fresh seeds (100-109). **Do not look at test results before the configuration is frozen.**

In [ ]:
frozen, msg = run_rq2.load_frozen_for_test(ROOT)   # checks version, copies and that it was selected on THIS run's dev results
print(msg)
if frozen is not None:
    watch("test", frozen=frozen, limit=LIMIT)
